In [1]:
!pip install pytorch-fid
!pip install scikit-learn torchmetrics scipy
!pip install lpips

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 121.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 46.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

In [2]:
# <<< THAY ĐỔI: Chạy dòng này trước tiên để cài đặt thư viện cần thiết
# !pip install lpips

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision.utils import make_grid
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import umap
from itertools import product
import lpips # <<< THAY ĐỔI: Import thư viện lpips

# ==============================================================================
# 1. Cấu hình và Thiết lập (Configuration and Setup)
# ==============================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR = "results_wae_annealing_v4_lpips" # <<< THAY ĐỔI: Đổi tên thư mục kết quả
os.makedirs(SAVE_DIR, exist_ok=True)

config = {
    "latent_dim": 32,
    "n_classes": 10,
    "batch_size": 128,
    "epochs": 50,
    "lr": 1e-3,
    "rho_prior": 0.7,
    "epsilon": 1e-8,
    # --- Cấu hình cho Loss ---
    "bce_weight": 0.3,      # <<< THAY ĐỔI: Trọng số cho BCE loss
    "lpips_weight": 0.7,    # <<< THAY ĐỔI: Trọng số cho LPIPS loss
    # --- Cấu hình cho Annealing ---
    "sup_mmd_weight": 20.0,
    "unsup_mmd_weight": 50.0,
    "anneal_epochs": 20,
}

# ==============================================================================
# 2. Các hàm tiện ích (Không thay đổi)
# ==============================================================================
def sample_uniform_sphere(n_samples, dim, device=DEVICE):
    eta = torch.randn(n_samples, dim, device=device)
    return F.normalize(eta, p=2, dim=1)

def mobius_reparam(eps, mu, rho):
    rho = rho.unsqueeze(-1) if rho.dim() == 1 else rho
    eps_mu_dot = torch.sum(eps * mu, dim=1, keepdim=True)
    numerator = (1 - rho**2) * eps + 2 * rho**2 * mu + 2 * rho * eps_mu_dot * mu
    denominator = 1 + 2 * rho * eps_mu_dot + rho**2
    z = numerator / (denominator + config["epsilon"])
    return F.normalize(z, p=2, dim=1)

def rbf_kernel(x, y, sigma):
    dist_sq = 2 - 2 * (x @ y.t())
    return torch.exp(-dist_sq / (2 * sigma**2 + config["epsilon"]))

def mmd_loss(q_samples, p_samples, sigma=None):
    if q_samples.shape[0] < 2 or p_samples.shape[0] < 2:
        return torch.tensor(0.0, device=DEVICE)
    if sigma is None:
        with torch.no_grad():
            dists = torch.pdist(torch.cat([q_samples, p_samples], dim=0))
            sigma = dists.median()
    k_qq = rbf_kernel(q_samples, q_samples, sigma).mean()
    k_pp = rbf_kernel(p_samples, p_samples, sigma).mean()
    k_qp = rbf_kernel(q_samples, p_samples, sigma).mean()
    return k_qq + k_pp - 2 * k_qp

# ==============================================================================
# 3. Kiến trúc Mô hình (Không thay đổi)
# ==============================================================================
class EncoderCNN(nn.Module):
    def __init__(self, latent_dim):
        super(EncoderCNN, self).__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU(True),
        )
        self.fc_block = nn.Sequential(nn.Flatten(), nn.Linear(128 * 4 * 4, 256), nn.ReLU(True))
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_s = nn.Linear(256, 1)
    def forward(self, x):
        x = self.conv_block(x)
        x = self.fc_block(x)
        return self.fc_mu(x), self.fc_s(x)

class DecoderCNN(nn.Module):
    def __init__(self, latent_dim):
        super(DecoderCNN, self).__init__()
        self.fc_block = nn.Sequential(nn.Linear(latent_dim, 256), nn.ReLU(True), nn.Linear(256, 128 * 4 * 4), nn.ReLU(True))
        self.deconv_block = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1), nn.Sigmoid()
        )
    def forward(self, z):
        x = self.fc_block(z)
        x = x.view(-1, 128, 4, 4)
        return self.deconv_block(x)

class SphericalWAE_Supervised(nn.Module):
    def __init__(self, latent_dim, n_classes):
        super(SphericalWAE_Supervised, self).__init__()
        self.encoder = EncoderCNN(latent_dim)
        self.decoder = DecoderCNN(latent_dim)
        self.prior_mus = nn.Parameter(torch.randn(n_classes, latent_dim))
        self.rho_p = config["rho_prior"]
    def encode_to_distribution(self, x):
        mu_q_unnormalized, s_q = self.encoder(x)
        mu_q = F.normalize(mu_q_unnormalized, p=2, dim=1)
        rho_q = torch.sigmoid(s_q).squeeze(-1) * (1 - config["epsilon"])
        return mu_q, rho_q
    def forward(self, x):
        mu_q, rho_q = self.encode_to_distribution(x)
        eps = sample_uniform_sphere(x.shape[0], config["latent_dim"], device=x.device)
        z_q = mobius_reparam(eps, mu_q, rho_q)
        x_hat = self.decoder(z_q)
        return x_hat, z_q

# ==============================================================================
# 4. Hàm tính toán Mất mát (<<< PHẦN CHỈNH SỬA CHÍNH)
# ==============================================================================
def calculate_loss(x, y, x_hat, z_q, model, loss_fn_vgg, sup_mmd_weight, unsup_mmd_weight):
    # --- Thành phần Tái tạo ---
    bce_loss = F.binary_cross_entropy(x_hat, x, reduction='mean')

    # Chuyển đổi thang đo của ảnh từ [0, 1] sang [-1, 1] cho LPIPS
    x_rescaled = (x * 2) - 1
    x_hat_rescaled = (x_hat * 2) - 1
    # LPIPS yêu cầu ảnh 3 kênh, ta lặp lại kênh màu xám 3 lần
    x_rescaled_rgb = x_rescaled.repeat(1, 3, 1, 1)
    x_hat_rescaled_rgb = x_hat_rescaled.repeat(1, 3, 1, 1)

    lpips_loss = loss_fn_vgg(x_hat_rescaled_rgb, x_rescaled_rgb).mean()

    # Kết hợp hai loss tái tạo
    recon_loss = config["bce_weight"] * bce_loss + config["lpips_weight"] * lpips_loss

    # --- Các thành phần MMD (giữ nguyên) ---
    supervised_mmd_loss = 0.0
    normalized_prior_mus = F.normalize(model.prior_mus, p=2, dim=1)
    for c in range(config["n_classes"]):
        class_mask = (y == c)
        if class_mask.sum() > 1:
            supervised_mmd_loss += mmd_loss(z_q[class_mask],
                                            mobius_reparam(sample_uniform_sphere(class_mask.sum(), config["latent_dim"]),
                                                           normalized_prior_mus[c].expand(class_mask.sum(), -1),
                                                           torch.full((class_mask.sum(),), model.rho_p, device=DEVICE)))
    supervised_mmd_loss /= config["n_classes"]

    random_classes = torch.randint(0, config["n_classes"], (x.size(0),), device=DEVICE)
    z_p_unsupervised = mobius_reparam(sample_uniform_sphere(x.size(0), config["latent_dim"]),
                                      normalized_prior_mus[random_classes],
                                      torch.full((x.size(0),), model.rho_p, device=DEVICE))
    unsupervised_mmd_loss = mmd_loss(z_q, z_p_unsupervised)

    # --- Loss tổng hợp ---
    total_loss = recon_loss + \
                 (sup_mmd_weight * supervised_mmd_loss) + \
                 (unsup_mmd_weight * unsupervised_mmd_loss)

    return total_loss, recon_loss, supervised_mmd_loss, unsupervised_mmd_loss

# ==============================================================================
# 5. Vòng lặp Huấn luyện
# ==============================================================================
def train_epoch(model, train_loader, optimizer, epoch, scheduler, loss_fn_vgg): # <<< THAY ĐỔI: Thêm loss_fn_vgg
    model.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['epochs']}")
    loss_acc, recon_acc, sup_mmd_acc, unsup_mmd_acc = 0.0, 0.0, 0.0, 0.0

    anneal_rate = min(1.0, (epoch + 1) / config["anneal_epochs"])
    current_sup_weight = config["sup_mmd_weight"] * anneal_rate
    current_unsup_weight = config["unsup_mmd_weight"] * anneal_rate

    for data, labels in pbar:
        data, labels = data.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()

        x_hat, z_q = model(data)

        # <<< THAY ĐỔI: Truyền loss_fn_vgg vào hàm loss
        loss, recon, sup_mmd, unsup_mmd = calculate_loss(data, labels, x_hat, z_q, model, loss_fn_vgg, current_sup_weight, current_unsup_weight)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        loss_acc += loss.item()
        recon_acc += recon.item()
        sup_mmd_acc += sup_mmd.item()
        unsup_mmd_acc += unsup_mmd.item()

        pbar.set_postfix({
            "Loss": f"{loss.item():.3f}", "Recon": f"{recon.item():.3f}",
            "SupMMD": f"{sup_mmd.item():.4f}", "UnsupMMD": f"{unsup_mmd.item():.4f}",
            "λ": f"{current_sup_weight:.2f}", "γ": f"{current_unsup_weight:.2f}"
        })

    scheduler.step()
    n_batches = len(train_loader)
    print(f"====> Epoch {epoch+1} Avg Loss: Total={loss_acc/n_batches:.4f}, Recon={recon_acc/n_batches:.4f}, SupMMD={sup_mmd_acc/n_batches:.4f}, UnsupMMD={unsup_mmd_acc/n_batches:.4f}")
    return loss_acc/n_batches, recon_acc/n_batches, sup_mmd_acc/n_batches, unsup_mmd_acc/n_batches

# ==============================================================================
# 6. Trực quan hóa và Hàm chính
# ==============================================================================
def plot_random_samples_from_priors(model, save_dir="."):
    print("Bắt đầu sinh ảnh ngẫu nhiên từ các tiên nghiệm...")
    model.eval()
    n_classes, latent_dim = config["n_classes"], config["latent_dim"]
    fig, axes = plt.subplots(n_classes, 8, figsize=(8 * 1.5, n_classes * 1.5))
    with torch.no_grad():
        normalized_prior_mus = F.normalize(model.prior_mus, p=2, dim=1)
        for c in range(n_classes):
            mu_p = normalized_prior_mus[c].expand(8, -1)
            rho_p = torch.full((8,), model.rho_p, device=DEVICE)
            eps = sample_uniform_sphere(8, latent_dim, device=DEVICE)
            z_p = mobius_reparam(eps, mu_p, rho_p)
            generated_images = model.decoder(z_p)
            for i, img in enumerate(generated_images):
                ax = axes[c, i]
                ax.imshow(img.cpu().squeeze(), cmap='gray')
                ax.axis('off')
                if i == 0:
                    ax.text(-10, 14, f'{c}', verticalalignment='center', horizontalalignment='right', fontsize=12, fontweight='bold')
    plt.suptitle("Ảnh sinh ngẫu nhiên từ các thành phần Tiên nghiệm (Annealing)")
    plt.savefig(f'{save_dir}/random_samples_from_priors.png')
    plt.close(fig)

def from_latent(net, vec):
    with torch.no_grad():
        net.eval()
        vec_tensor = torch.from_numpy(vec).unsqueeze(0).to(DEVICE).float()
        vec_tensor_normalized = F.normalize(vec_tensor, p=2, dim=1)
        return net.decoder(vec_tensor_normalized).cpu().numpy().reshape(28, 28)

def get_sampling_grid(net, grid):
    base = torch.randn(config["latent_dim"] - 2)
    image_list = [torch.from_numpy(from_latent(net, np.hstack([vec_2d, base.numpy()]))) for vec_2d in grid]
    return torch.stack(image_list).unsqueeze(1)

def plot_grid_samples(model, save_dir="."):
    print("Bắt đầu sinh ảnh từ lưới phẳng (phương pháp so sánh)...")
    grid_points = list(product(np.linspace(-1.5, 1.5, 8), np.linspace(-1.5, 1.5, 8)))
    results = get_sampling_grid(model, grid_points)
    fig = plt.figure(figsize=(10, 10))
    img_grid = make_grid(results, nrow=8)
    plt.imshow(img_grid.permute(1, 2, 0))
    plt.title("Ảnh sinh ra từ Lưới Phẳng (Grid Sampling)")
    plt.axis('off')
    plt.savefig(f'{save_dir}/grid_samples.png')
    plt.close(fig)

def plot_results(history, model, test_loader, save_dir="."):
    print("Bắt đầu vẽ biểu đồ và trực quan hóa kết quả...")
    fig = plt.figure(figsize=(12, 8))
    plt.plot([h[0] for h in history], label='Total Loss')
    plt.plot([h[1] for h in history], label='Reconstruction Loss')
    plt.plot([h[2] for h in history], label='Supervised MMD Loss')
    plt.plot([h[3] for h in history], label='Unsupervised MMD Loss')
    plt.title('Lịch sử Huấn luyện (WAE with Annealing)')
    plt.xlabel('Epoch'); plt.ylabel('Loss Value'); plt.legend(); plt.grid(True)
    plt.savefig(f'{save_dir}/loss_history.png'); plt.close(fig)

    model.eval()
    with torch.no_grad():
        data, _ = next(iter(test_loader)); data = data.to(DEVICE); x_hat, _ = model(data)
        fig = plt.figure(figsize=(20, 4))
        n = 10
        for i in range(n):
            ax = plt.subplot(2, n, i + 1); plt.imshow(data[i].cpu().squeeze(), cmap='gray'); plt.title("Gốc"); ax.axis('off')
            ax = plt.subplot(2, n, i + 1 + n); plt.imshow(x_hat[i].cpu().squeeze(), cmap='gray'); plt.title("Tái tạo"); ax.axis('off')
        plt.savefig(f'{save_dir}/reconstructions.png'); plt.close(fig)

        print("Bắt đầu chiếu không gian ẩn bằng UMAP...")
        latent_mus, labels = [], []
        for data, target in tqdm(test_loader, desc="Encoding test set for UMAP"):
            mu_q, _ = model.encode_to_distribution(data.to(DEVICE))
            latent_mus.append(mu_q.cpu().numpy()); labels.append(target.numpy())

        latent_mus = np.concatenate(latent_mus, axis=0); labels = np.concatenate(labels, axis=0)

        reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
        embedding = reducer.fit_transform(latent_mus)

        prior_mus_np = F.normalize(model.prior_mus, p=2, dim=1).cpu().detach().numpy()
        prior_embedding = reducer.transform(prior_mus_np)

        fig = plt.figure(figsize=(12, 10))
        scatter = plt.scatter(embedding[:, 0], embedding[:, 1], c=labels, cmap='Spectral', s=5, alpha=0.7)
        plt.scatter(prior_embedding[:, 0], prior_embedding[:, 1], c=range(10), cmap='Spectral', marker='*', s=500, edgecolor='black', label='Prior Centers')
        plt.title('Không gian ẩn (Latent Space) - UMAP (Annealing)')
        plt.legend(handles=scatter.legend_elements(num=10)[0], labels=list(range(10)))
        plt.colorbar(scatter); plt.savefig(f'{save_dir}/latent_space_umap.png'); plt.close(fig)

    plot_random_samples_from_priors(model, save_dir=save_dir)
    plot_grid_samples(model, save_dir=save_dir)

In [3]:
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

num_workers = 2 if os.name == 'nt' else 4
train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True, pin_memory=True, num_workers=num_workers)
test_loader = DataLoader(test_dataset, batch_size=config["batch_size"], shuffle=False, pin_memory=True, num_workers=num_workers)

model = SphericalWAE_Supervised(latent_dim=config["latent_dim"], n_classes=config["n_classes"]).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=config["lr"])
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)

# <<< THAY ĐỔI: Khởi tạo LPIPS loss function
loss_fn_vgg = lpips.LPIPS(net='vgg').to(DEVICE)

print(f"Bắt đầu huấn luyện trên thiết bị: {DEVICE} với kiến trúc WAE, Annealing và LPIPS Loss")
print(f"Cấu hình: {config}")

history = []
for epoch in range(config["epochs"]):
    # <<< THAY ĐỔI: Truyền loss_fn_vgg vào train_epoch
    avg_losses = train_epoch(model, train_loader, optimizer, epoch, scheduler, loss_fn_vgg)
    history.append(avg_losses)

print("Hoàn tất huấn luyện!")

torch.save(model.state_dict(), f'{SAVE_DIR}/spcauchy_wae_annealing_lpips.pth')
print(f"Đã lưu mô hình đã huấn luyện vào '{SAVE_DIR}/spcauchy_wae_annealing_lpips.pth'")

plot_results(history, model, test_loader, save_dir=SAVE_DIR)

100%|██████████| 9.91M/9.91M [00:00<00:00, 15.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 508kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.03MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.28MB/s]
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100%|██████████| 528M/528M [00:07<00:00, 77.3MB/s]


Loading model from: /usr/local/lib/python3.11/dist-packages/lpips/weights/v0.1/vgg.pth
Bắt đầu huấn luyện trên thiết bị: cuda với kiến trúc WAE, Annealing và LPIPS Loss
Cấu hình: {'latent_dim': 32, 'n_classes': 10, 'batch_size': 128, 'epochs': 50, 'lr': 0.001, 'rho_prior': 0.7, 'epsilon': 1e-08, 'bce_weight': 0.3, 'lpips_weight': 0.7, 'sup_mmd_weight': 20.0, 'unsup_mmd_weight': 50.0, 'anneal_epochs': 20}


Epoch 1/50: 100%|██████████| 469/469 [00:45<00:00, 10.35it/s, Loss=0.280, Recon=0.101, SupMMD=0.1336, UnsupMMD=0.0181, λ=1.00, γ=2.50]


...

====> Epoch 49 Avg Loss: Total=1.8083, Recon=0.0612, SupMMD=0.0749, UnsupMMD=0.0050


Epoch 50/50: 100%|██████████| 469/469 [00:44<00:00, 10.65it/s, Loss=2.160, Recon=0.060, SupMMD=0.0963, UnsupMMD=0.0035, λ=20.00, γ=50.00]


====> Epoch 50 Avg Loss: Total=1.8064, Recon=0.0610, SupMMD=0.0748, UnsupMMD=0.0050
Hoàn tất huấn luyện!
Đã lưu mô hình đã huấn luyện vào 'results_wae_annealing_v4_lpips/spcauchy_wae_annealing_lpips.pth'
Bắt đầu vẽ biểu đồ và trực quan hóa kết quả...
Bắt đầu chiếu không gian ẩn bằng UMAP...


Encoding test set for UMAP: 100%|██████████| 79/79 [00:01<00:00, 52.10it/s]
/usr/local/lib/python3.11/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Bắt đầu sinh ảnh ngẫu nhiên từ các tiên nghiệm...
Bắt đầu sinh ảnh từ lưới phẳng (phương pháp so sánh)...


In [4]:
# ==============================================================================
# Sửa lỗi và Bổ sung hàm vẽ Slerp
# ==============================================================================

def slerp(p0, p1, t, epsilon=1e-8):
    """
    Nội suy trên mặt cầu (Spherical Linear Interpolation).
    p0, p1: các vector bắt đầu và kết thúc (đã được chuẩn hóa).
    t: một giá trị hoặc tensor chứa các giá trị từ 0 đến 1.
    """
    # Tính góc giữa hai vector
    omega = torch.acos(torch.dot(p0, p1).clamp(-1, 1))
    sin_omega = torch.sin(omega)

    # Nếu hai vector quá gần nhau, trả về vector ban đầu để tránh chia cho 0
    if sin_omega.item() < epsilon:
        # Mở rộng p0 để có cùng số chiều với output mong muốn khi t là một tensor
        return p0.unsqueeze(0).expand(len(t), -1)

    # Đảm bảo t có cùng device với các vector
    t = t.to(p0.device)

    # Công thức Slerp
    a = torch.sin((1.0 - t) * omega) / sin_omega
    b = torch.sin(t * omega) / sin_omega

    # unsqueeze() để thực hiện phép nhân broadcast đúng cách
    return a.unsqueeze(-1) * p0.unsqueeze(0) + b.unsqueeze(-1) * p1.unsqueeze(0)


def plot_slerp(model, save_dir=".", num_steps=10):
    """
    Vẽ và lưu ảnh được tạo ra từ phép nội suy Slerp giữa các cặp tiên nghiệm.
    """
    print("Bắt đầu sinh ảnh nội suy Slerp...")
    model.eval()

    # Chọn một vài cặp chữ số thú vị để nội suy
    interpolation_pairs = [(1, 7), (3, 5), (2, 8), (4, 9)]
    num_pairs = len(interpolation_pairs)

    fig, axes = plt.subplots(num_pairs, num_steps, figsize=(num_steps * 1.5, num_pairs * 1.5))

    with torch.no_grad():
        t_values = torch.linspace(0, 1, num_steps)

        for row, (digit_a, digit_b) in enumerate(interpolation_pairs):
            # LẤY VECTOR VÀ SỬA LỖI: Thêm `dim=0`
            mu_a = F.normalize(model.prior_mus[digit_a].detach(), dim=0)
            mu_b = F.normalize(model.prior_mus[digit_b].detach(), dim=0)

            # Thực hiện Slerp
            interpolated_z = slerp(mu_a, mu_b, t_values).to(DEVICE)

            # Giải mã các vector z trung gian
            generated_images = model.decoder(interpolated_z)

            # Vẽ các ảnh
            for col, img in enumerate(generated_images):
                ax = axes[row, col]
                ax.imshow(img.cpu().squeeze(), cmap='gray')
                ax.axis('off')

                # Ghi nhãn cho ảnh đầu và cuối
                if col == 0:
                    ax.set_title(f'{digit_a}')
                if col == num_steps - 1:
                    ax.set_title(f'{digit_b}')

    plt.suptitle("Nội suy Slerp giữa các Tiên nghiệm (Slerp Interpolation)")
    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Điều chỉnh layout để title không bị đè
    plt.savefig(f'{save_dir}/slerp_interpolation.png')
    plt.close(fig)
    print(f"Đã lưu ảnh Slerp vào '{save_dir}/slerp_interpolation.png'")

In [5]:
# THÊM LỆNH GỌI HÀM MỚI Ở ĐÂY
plot_slerp(model, save_dir=SAVE_DIR)

Bắt đầu sinh ảnh nội suy Slerp...
Đã lưu ảnh Slerp vào 'results_wae_annealing_v4_lpips/slerp_interpolation.png'


In [6]:
# ==============================================================================
#                 PHẦN ĐÁNH GIÁ MÔ HÌNH TOÀN DIỆN (TỔNG HỢP)
# ==============================================================================
# Cell này tổng hợp tất cả các bước tính toán metrics vào một nơi duy nhất.
# Bao gồm:
# 1. Metrics Tái tạo & Sinh ảnh: FID, SSIM, PSNR, LPIPS
# 2. Metrics Phân cụm: ACC, NMI, ARI
# ------------------------------------------------------------------------------

# --- BƯỚC 1: CÀI ĐẶT VÀ IMPORT CÁC THƯ VIỆN CẦN THIẾT ---
# Đảm bảo các thư viện đã được cài đặt từ các cell trước
# !pip install pytorch-fid scikit-learn torchmetrics scipy lpips

import os
import torch
import torchvision
import numpy as np
from tqdm import tqdm
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score, accuracy_score, adjusted_rand_score # Thêm ARI
from pytorch_fid import fid_score
import lpips

# TorchMetrics
from torchmetrics.image import StructuralSimilarityIndexMeasure, PeakSignalNoiseRatio

# Tái sử dụng các phần đã có từ code của bạn
# (Giả định các lớp model, hàm, và config đã được định nghĩa ở trên)

# ==============================================================================
# --- BƯỚC 2: CẤU HÌNH VÀ TẢI MODEL ---
# ==============================================================================
print(" Bắt đầu quá trình đánh giá toàn diện ".center(80, "="))

# Đường dẫn đến file .pth chứa trọng số mô hình đã huấn luyện
MODEL_PATH = f'{SAVE_DIR}/spcauchy_wae_annealing_lpips.pth'

# Tải lại kiến trúc model
model = SphericalWAE_Supervised(
    latent_dim=config["latent_dim"],
    n_classes=config["n_classes"]
).to(DEVICE)

# Tải trọng số đã huấn luyện
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval() # Chuyển model sang chế độ đánh giá

# Tải bộ dữ liệu test
transform = transforms.Compose([transforms.ToTensor()])
test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=config["batch_size"], shuffle=False)

print(f"Đã tải thành công model từ '{MODEL_PATH}' và bộ dữ liệu test.")

# ==============================================================================
# --- BƯỚC 3: TÍNH TOÁN ĐỒNG THỜI CÁC METRICS ---
# ==============================================================================
print("\n" + " Bắt đầu tính toán các metrics ".center(80, "-"))

# --- Khởi tạo các đối tượng metrics và biến lưu trữ ---
# Metrics tái tạo
ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(DEVICE)
psnr_metric = PeakSignalNoiseRatio(data_range=1.0).to(DEVICE)
lpips_metric = lpips.LPIPS(net='vgg').to(DEVICE)
total_lpips_score = 0.0

# Biến lưu trữ cho metrics phân cụm
all_latents = []
all_labels = []

# --- Vòng lặp tính toán trên toàn bộ test set ---
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Đang xử lý bộ test"):
        images = images.to(DEVICE)

        # --- 1. Tái tạo và Mã hóa ---
        reconstructed_images, mu_q = model(images)
        # Lấy mu_q làm vector đại diện cho không gian ẩn
        mu_q_unnormalized, _ = model.encoder(images)
        latent_vectors = F.normalize(mu_q_unnormalized, p=2, dim=1)

        # --- 2. Cập nhật Metrics Tái tạo ---
        ssim_metric.update(reconstructed_images, images)
        psnr_metric.update(reconstructed_images, images)

        # Tính LPIPS (yêu cầu ảnh 3 kênh và thang đo [-1, 1])
        images_lpips = (images * 2 - 1).repeat(1, 3, 1, 1)
        recon_lpips = (reconstructed_images * 2 - 1).repeat(1, 3, 1, 1)
        total_lpips_score += lpips_metric(images_lpips, recon_lpips).sum().item()

        # --- 3. Thu thập dữ liệu cho Metrics Phân cụm ---
        all_latents.append(latent_vectors.cpu().numpy())
        all_labels.append(labels.numpy())

# --- Lấy kết quả cuối cùng cho Metrics Tái tạo ---
final_ssim = ssim_metric.compute().item()
final_psnr = psnr_metric.compute().item()
final_lpips = total_lpips_score / len(test_dataset)

print("Hoàn tất tính toán metrics tái tạo.")

# --- Tính toán Metrics Phân cụm ---
print("Bắt đầu tính toán metrics phân cụm...")
latents_np = np.concatenate(all_latents, axis=0)
labels_np = np.concatenate(all_labels, axis=0)

print(f"Đã trích xuất {len(latents_np)} vector ẩn. Bắt đầu chạy K-Means...")
kmeans = KMeans(n_clusters=config["n_classes"], random_state=42, n_init='auto')
cluster_preds = kmeans.fit_predict(latents_np)

# Tính NMI
nmi_score = normalized_mutual_info_score(labels_np, cluster_preds)
# Tính ARI (MỚI)
ari_score = adjusted_rand_score(labels_np, cluster_preds)

# Tính ACC
contingency_matrix = np.zeros((config["n_classes"], config["n_classes"]), dtype=np.int64)
for i in range(len(labels_np)):
    contingency_matrix[cluster_preds[i], labels_np[i]] += 1
row_ind, col_ind = linear_sum_assignment(-contingency_matrix)
acc_score = contingency_matrix[row_ind, col_ind].sum() / len(labels_np)

print("Hoàn tất tính toán metrics phân cụm.")

# --- Tính toán FID (giữ nguyên từ code trước của bạn) ---
# Lưu ý: Phần này yêu cầu lưu ảnh ra đĩa và có thể mất thời gian.
# Bạn có thể comment phần này nếu đã chạy và có kết quả.
print("\nBắt đầu tính toán FID (có thể mất vài phút)...")
NUM_IMAGES_FOR_FID = 10000
REAL_IMAGES_DIR = "fid_images/real"
GEN_IMAGES_DIR = "fid_images/generated"
os.makedirs(REAL_IMAGES_DIR, exist_ok=True)
os.makedirs(GEN_IMAGES_DIR, exist_ok=True)

# Lưu ảnh thật
saved_count = 0
for images, _ in tqdm(test_loader, desc="Lưu ảnh thật cho FID"):
    for i in range(images.size(0)):
        if saved_count >= NUM_IMAGES_FOR_FID: break
        image_rgb = images[i].repeat(3, 1, 1)
        torchvision.utils.save_image(image_rgb, os.path.join(REAL_IMAGES_DIR, f"real_{saved_count}.png"))
        saved_count += 1
    if saved_count >= NUM_IMAGES_FOR_FID: break

# Sinh và lưu ảnh giả
generated_count = 0
with torch.no_grad():
    while generated_count < NUM_IMAGES_FOR_FID:
        num_to_gen = min(config["batch_size"], NUM_IMAGES_FOR_FID - generated_count)
        random_classes = torch.randint(0, config["n_classes"], (num_to_gen,), device=DEVICE)
        normalized_prior_mus = F.normalize(model.prior_mus, p=2, dim=1)
        eps = sample_uniform_sphere(num_to_gen, config["latent_dim"], device=DEVICE)
        z_p = mobius_reparam(eps, normalized_prior_mus[random_classes], torch.full((num_to_gen,), model.rho_p, device=DEVICE))
        generated_images = model.decoder(z_p)
        for i in range(generated_images.size(0)):
            image_rgb = generated_images[i].cpu().repeat(3, 1, 1)
            torchvision.utils.save_image(image_rgb, os.path.join(GEN_IMAGES_DIR, f"gen_{generated_count}.png"))
            generated_count += 1

fid_value = fid_score.calculate_fid_given_paths(
    paths=[REAL_IMAGES_DIR, GEN_IMAGES_DIR], batch_size=50, device=DEVICE, dims=2048
)
print(f"Hoàn tất tính toán FID: {fid_value:.4f}")


# ==============================================================================
# --- BƯỚC 4: IN BẢNG TỔNG KẾT ---
# ==============================================================================
print("\n" + " BẢNG TỔNG KẾT METRICS ".center(80, "="))
print(f"| {'Metric':<28} | {'Giá trị':<15} | {'Diễn giải':<25} |")
print(f"|{'-'*30}|{'-'*17}|{'-'*27}|")
print(f"| {'--- CHẤT LƯỢNG SINH ẢNH ---':<76} |")
print(f"| {'FID (Fréchet Inception Dist.)':<28} | {fid_value:<15.4f} | {'Càng thấp càng tốt':<25} |")
print(f"| {'LPIPS (Perceptual Similarity)':<28} | {final_lpips:<15.4f} | {'Càng thấp càng tốt':<25} |")
print(f"|{'-'*30}|{'-'*17}|{'-'*27}|")
print(f"| {'--- CHẤT LƯỢNG TÁI TẠO ---':<76} |")
print(f"| {'SSIM (Structural Similarity)':<28} | {final_ssim:<15.4f} | {'Càng gần 1 càng tốt':<25} |")
print(f"| {'PSNR (Peak Signal-to-Noise)':<28} | {final_psnr:<15.4f} | {'Càng cao càng tốt':<25} |")
print(f"|{'-'*30}|{'-'*17}|{'-'*27}|")
print(f"| {'--- CHẤT LƯỢNG PHÂN CỤM ---':<76} |")
print(f"| {'ACC (Clustering Accuracy)':<28} | {acc_score:<15.4f} | {'Càng gần 1 càng tốt':<25} |")
print(f"| {'NMI (Normalized Mutual Info)':<28} | {nmi_score:<15.4f} | {'Càng gần 1 càng tốt':<25} |")
print(f"| {'ARI (Adjusted Rand Index)':<28} | {ari_score:<15.4f} | {'Càng gần 1 càng tốt':<25} |")
print("=" * 80)


===================== Bắt đầu quá trình đánh giá toàn diện =====================
Đã tải thành công model từ 'results_wae_annealing_v4_lpips/spcauchy_wae_annealing_lpips.pth' và bộ dữ liệu test.

------------------------ Bắt đầu tính toán các metrics -------------------------
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /usr/local/lib/python3.11/dist-packages/lpips/weights/v0.1/vgg.pth


Đang xử lý bộ test: 100%|██████████| 79/79 [00:03<00:00, 21.77it/s]


Hoàn tất tính toán metrics tái tạo.
Bắt đầu tính toán metrics phân cụm...
Đã trích xuất 10000 vector ẩn. Bắt đầu chạy K-Means...
Hoàn tất tính toán metrics phân cụm.

Bắt đầu tính toán FID (có thể mất vài phút)...


Lưu ảnh thật cho FID:  99%|█████████▊| 78/79 [00:06<00:00, 11.84it/s]
Downloading: "https://github.com/mseitzer/pytorch-fid/releases/download/fid_weights/pt_inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/pt_inception-2015-12-05-6726825d.pth
100%|██████████| 91.2M/91.2M [00:00<00:00, 227MB/s]
100%|██████████| 200/200 [00:39<00:00,  5.04it/s]


Hoàn tất tính toán FID: 15.5977

============================ BẢNG TỔNG KẾT METRICS =============================
| Metric                       | Giá trị         | Diễn giải                 |
|------------------------------|-----------------|---------------------------|
| --- CHẤT LƯỢNG SINH ẢNH ---                                                  |
| FID (Fréchet Inception Dist.) | 15.5977         | Càng thấp càng tốt        |
| LPIPS (Perceptual Similarity) | 0.0454          | Càng thấp càng tốt        |
|------------------------------|-----------------|---------------------------|
| --- CHẤT LƯỢNG TÁI TẠO ---                                                   |
| SSIM (Structural Similarity) | 0.8717          | Càng gần 1 càng tốt       |
| PSNR (Peak Signal-to-Noise)  | 18.1812         | Càng cao càng tốt         |
|------------------------------|-----------------|---------------------------|
| --- CHẤT LƯỢNG PHÂN CỤM ---                                                  |
| ACC (Cl

---

## BaseLine VAE, S-VAE, WAE-MMD, VADE

In [5]:
!pip install git+https://github.com/nicola-decao/s-vae-pytorch.git

  Cloning https://github.com/nicola-decao/s-vae-pytorch.git to /tmp/pip-req-build-8_3jcg94
  Running command git clone --filter=blob:none --quiet https://github.com/nicola-decao/s-vae-pytorch.git /tmp/pip-req-build-8_3jcg94
  Resolved https://github.com/nicola-decao/s-vae-pytorch.git to commit 671c46c0772a2c92a602252311864c48c816bc44
  Preparing metadata (setup.py) ... done
  Created wheel for hyperspherical_vae: filename=hyperspherical_vae-0.1.1-py3-none-any.whl size=8360 sha256=de201796ac86fb21855430c5f96b22ed20b1dcce4518255cebc96761407e12fc
  Stored in directory: /tmp/pip-ephem-wheel-cache-f4c3mrfc/wheels/f7/06/8e/5c49623ea2d57cd6045356f49435e3e99b49ac19c47c352345
Successfully built hyperspherical_vae


In [14]:
# ==============================================================================
#      NOTEBOOK TRIỂN KHAI THÍ NGHIỆM SO SÁNH VÀ ĐÁNH GIÁ BASELINES
# ==============================================================================
# Hướng dẫn:
# 1. Chạy cell này trong một môi trường có GPU (ví dụ: Google Colab, Kaggle).
# 2. Quá trình huấn luyện và đánh giá tất cả các model có thể mất nhiều thời gian.
# 3. Kết quả cuối cùng sẽ là một bảng so sánh ở định dạng Markdown.
# ------------------------------------------------------------------------------

# --- BƯỚC 1: CÀI ĐẶT VÀ IMPORT CÁC THƯ VIỆN CẦN THIẾT ---
print(">>> Bước 1: Cài đặt và import thư viện...")
# Cài đặt các thư viện cần thiết một cách thầm lặng
import os
os.environ['PYTHONWARNINGS'] = 'ignore'
try:
    import pytorch_fid
    import lpips
    import torchmetrics
    from hyperspherical_vae.distributions import VonMisesFisher, HypersphericalUniform
except ImportError:
    print("Đang cài đặt các thư viện cần thiết...")
    # SỬA LỖI 1: Cài đặt hyperspherical-vae từ GitHub
    os.system('pip install -q git+https://github.com/nicola-decao/s-vae-pytorch.git')
    os.system('pip install -q pytorch-fid lpips torchmetrics scikit-learn scipy umap-learn')
    print("Cài đặt hoàn tất.")
    from hyperspherical_vae.distributions import VonMisesFisher, HypersphericalUniform


import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision.utils import save_image
import numpy as np
import pandas as pd
from tqdm import tqdm
import warnings

# Imports cho Evaluation
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score, accuracy_score, adjusted_rand_score
from pytorch_fid import fid_score
import lpips as lpips_lib
from torchmetrics.image import StructuralSimilarityIndexMeasure, PeakSignalNoiseRatio

warnings.filterwarnings('ignore')

# ==============================================================================
# --- BƯỚC 2: CẤU HÌNH CHUNG CHO THÍ NGHIỆM ---
# ==============================================================================
print("\n>>> Bước 2: Thiết lập cấu hình chung...")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS_DIR = "baseline_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

CONFIG = {
    "latent_dim": 32,
    "n_classes": 10,
    "batch_size": 128,
    "epochs": 50, # Giảm xuống để chạy nhanh hơn, khuyến nghị 50 cho kết quả tốt nhất
    "lr": 1e-3,
}

# Tải dữ liệu MNIST
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True, pin_memory=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=CONFIG["batch_size"], shuffle=False, pin_memory=True, num_workers=2)

# ==============================================================================
# --- BƯỚC 3: ĐỊNH NGHĨA CÁC KIẾN TRÚC MODEL BASELINE ---
# ==============================================================================
print("\n>>> Bước 3: Định nghĩa kiến trúc các model baseline...")

# --- Các khối xây dựng chung ---
class EncoderCNN(nn.Module):
    def __init__(self, latent_dim, vae_mode=False):
        super(EncoderCNN, self).__init__()
        self.vae_mode = vae_mode
        self.conv_block = nn.Sequential(
            nn.Conv2d(1, 32, 4, 2, 1), nn.ReLU(True),
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(True),
            nn.Conv2d(64, 128, 4, 2, 1), nn.ReLU(True),
        )
        self.fc_block = nn.Sequential(nn.Flatten(), nn.Linear(128 * 3 * 3, 256), nn.ReLU(True))
        self.fc_out1 = nn.Linear(256, latent_dim) # mu cho VAE, z cho WAE
        if self.vae_mode:
            self.fc_out2 = nn.Linear(256, latent_dim) # log_var cho VAE

    def forward(self, x):
        x = self.conv_block(x)
        x = x.view(x.size(0), -1) # Flatten
        x = self.fc_block(x)
        out1 = self.fc_out1(x)
        if self.vae_mode:
            out2 = self.fc_out2(x)
            return out1, out2
        return out1

class DecoderCNN(nn.Module):
    def __init__(self, latent_dim):
        super(DecoderCNN, self).__init__()
        self.fc_block = nn.Sequential(nn.Linear(latent_dim, 256), nn.ReLU(True), nn.Linear(256, 128 * 7 * 7), nn.ReLU(True))
        self.deconv_block = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(True),
            nn.ConvTranspose2d(32, 1, 4, 2, 1), nn.Sigmoid()
        )
    def forward(self, z):
        x = self.fc_block(z)
        x = x.view(-1, 128, 7, 7)
        x = self.deconv_block(x)
        # Cắt bớt để có kích thước 28x28
        return x[:, :, :28, :28]

# --- Model 1: Vanilla VAE ---
class VAE(nn.Module):
    def __init__(self, latent_dim):
        super(VAE, self).__init__()
        self.encoder = EncoderCNN(latent_dim, vae_mode=True)
        self.decoder = DecoderCNN(latent_dim)

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, log_var = self.encoder(x)
        z = self.reparameterize(mu, log_var)
        return self.decoder(z), mu, log_var

    def loss_function(self, recon_x, x, mu, log_var):
        BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')
        KLD = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
        return BCE + KLD

# --- Model 2: WAE-MMD ---
def rbf_kernel(x, y, sigma=1.0):
    dist_sq = torch.cdist(x, y, p=2).pow(2)
    return torch.exp(-dist_sq / (2 * sigma**2))

def mmd_loss(q_samples, p_samples, sigma=1.0):
    k_qq = rbf_kernel(q_samples, q_samples, sigma).mean()
    k_pp = rbf_kernel(p_samples, p_samples, sigma).mean()
    k_qp = rbf_kernel(q_samples, p_samples, sigma).mean()
    return k_qq + k_pp - 2 * k_qp

class WAE_MMD(nn.Module):
    def __init__(self, latent_dim):
        super(WAE_MMD, self).__init__()
        self.encoder = EncoderCNN(latent_dim)
        self.decoder = DecoderCNN(latent_dim)
        self.latent_dim = latent_dim

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

    def loss_function(self, recon_x, x, z, mmd_weight=10.0):
        recon_loss = F.binary_cross_entropy(recon_x, x, reduction='mean')
        true_samples = torch.randn(z.size(0), self.latent_dim, device=z.device)
        mmd = mmd_loss(z, true_samples)
        return recon_loss + mmd_weight * mmd

# --- Model 3: S-VAE (Spherical VAE) ---
class S_VAE(nn.Module):
    def __init__(self, latent_dim):
        super(S_VAE, self).__init__()
        self.encoder = EncoderCNN(latent_dim + 1)
        self.decoder = DecoderCNN(latent_dim)
        self.latent_dim = latent_dim

    def forward(self, x):
        q_params = self.encoder(x)
        mu = F.normalize(q_params[:, :-1], p=2, dim=1) # Ensure mu is on the sphere
        kappa = F.softplus(q_params[:, -1].unsqueeze(-1)) + 1e-6

        q_z = VonMisesFisher(mu, kappa)
        # SỬA LỖI 2: Tham số đúng cho HypersphericalUniform là `d` (số chiều)
        p_z = HypersphericalUniform(self.latent_dim, device=DEVICE)

        z = q_z.rsample()
        recon_x = self.decoder(z)
        return recon_x, q_z, p_z

    def loss_function(self, recon_x, x, q_z, p_z):
        recon_loss = F.binary_cross_entropy(recon_x, x, reduction='sum')
        kld = torch.distributions.kl.kl_divergence(q_z, p_z).sum()
        return recon_loss + kld

# --- Model 4: VaDE (Variational Deep Embedding) ---
class VaDE(nn.Module):
    def __init__(self, latent_dim, n_classes):
        super(VaDE, self).__init__()
        self.encoder = EncoderCNN(latent_dim, vae_mode=True)
        self.decoder = DecoderCNN(latent_dim)
        self.latent_dim = latent_dim
        self.n_classes = n_classes

        # GMM parameters
        self.pi = nn.Parameter(torch.ones(n_classes) / n_classes, requires_grad=True)
        self.mu_c = nn.Parameter(torch.randn(n_classes, latent_dim), requires_grad=True)
        self.log_var_c = nn.Parameter(torch.randn(n_classes, latent_dim), requires_grad=True)

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, log_var = self.encoder(x)
        z = self.reparameterize(mu, log_var)
        return self.decoder(z), mu, log_var, z

    def loss_function(self, recon_x, x, mu, log_var, z):
        # Reconstruction Loss
        recon_loss = F.binary_cross_entropy(recon_x, x, reduction='sum')

        # GMM Prior Loss (KL Divergence)
        z_expanded = z.unsqueeze(1)
        mu_c_expanded = self.mu_c.unsqueeze(0)
        log_var_c_expanded = self.log_var_c.unsqueeze(0)

        # Calculate gamma (responsibilities)
        log_p_z_given_c = -0.5 * (
            torch.sum(log_var_c_expanded, dim=2) +
            torch.sum((z_expanded - mu_c_expanded).pow(2) / torch.exp(log_var_c_expanded), dim=2) +
            self.latent_dim * np.log(2 * np.pi)
        )
        log_p_c_p_z_given_c = F.log_softmax(self.pi, dim=0) + log_p_z_given_c
        gamma = F.softmax(log_p_c_p_z_given_c, dim=1)

        # Calculate KL Divergence part
        kl_div = 0.5 * torch.sum(gamma * (
            torch.sum(log_var_c_expanded - log_var.unsqueeze(1), dim=2) +
            torch.sum((torch.exp(log_var.unsqueeze(1)) + (mu.unsqueeze(1) - mu_c_expanded).pow(2)) / torch.exp(log_var_c_expanded), dim=2) -
            self.latent_dim
        ))
        kl_div -= torch.sum(gamma * torch.log(self.pi.unsqueeze(0) / (gamma + 1e-10)))

        return recon_loss + kl_div


# ==============================================================================
# --- BƯỚC 4: HÀM ĐÁNH GIÁ TOÀN DIỆN ---
# ==============================================================================
print("\n>>> Bước 4: Xây dựng hàm đánh giá toàn diện...")

@torch.no_grad()
def evaluate_model(model, model_name, test_loader, device, config):
    print(f"\n--- Bắt đầu đánh giá model: {model_name} ---")
    model.eval()
    ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
    psnr_metric = PeakSignalNoiseRatio(data_range=1.0).to(device)
    lpips_metric = lpips_lib.LPIPS(net='vgg').to(device)
    total_lpips_score = 0.0
    all_latents, all_labels = [], []

    for images, labels in tqdm(test_loader, desc=f"Đánh giá {model_name}"):
        images = images.to(device)

        if model_name == 'VAE' or model_name == 'VaDE':
            recon_images, mu, _ = model(images)[:3]
            latent_vectors = mu
        elif model_name == 'WAE-MMD':
            recon_images, z = model(images)
            latent_vectors = z
        elif model_name == 'S-VAE':
            recon_images, q_z, _ = model(images)
            latent_vectors = q_z.mean

        ssim_metric.update(recon_images, images)
        psnr_metric.update(recon_images, images)
        images_lpips = (images * 2 - 1).repeat(1, 3, 1, 1)
        recon_lpips = (recon_images * 2 - 1).repeat(1, 3, 1, 1)
        total_lpips_score += lpips_metric(images_lpips, recon_lpips).sum().item()
        all_latents.append(latent_vectors.cpu().numpy())
        all_labels.append(labels.numpy())

    final_ssim = ssim_metric.compute().item()
    final_psnr = psnr_metric.compute().item()
    final_lpips = total_lpips_score / len(test_dataset)

    latents_np = np.concatenate(all_latents, axis=0)
    labels_np = np.concatenate(all_labels, axis=0)

    if model_name == 'VaDE':
        cluster_centers = model.mu_c.detach().cpu().numpy()
        kmeans = KMeans(n_clusters=config["n_classes"], init=cluster_centers, n_init=1).fit(latents_np)
    else:
        kmeans = KMeans(n_clusters=config["n_classes"], random_state=42, n_init='auto').fit(latents_np)

    cluster_preds = kmeans.predict(latents_np)
    nmi_score = normalized_mutual_info_score(labels_np, cluster_preds)
    ari_score = adjusted_rand_score(labels_np, cluster_preds)

    contingency_matrix = np.zeros((10, 10), dtype=np.int64)
    for i in range(len(labels_np)): contingency_matrix[cluster_preds[i], labels_np[i]] += 1
    row_ind, col_ind = linear_sum_assignment(-contingency_matrix)
    acc_score = contingency_matrix[row_ind, col_ind].sum() / len(labels_np)

    print(f"Bắt đầu tính FID cho {model_name}...")
    FID_DIR = os.path.join(RESULTS_DIR, f"fid_images_{model_name}")
    REAL_IMG_DIR = os.path.join(FID_DIR, "real"); GEN_IMG_DIR = os.path.join(FID_DIR, "generated")
    os.makedirs(REAL_IMG_DIR, exist_ok=True); os.makedirs(GEN_IMG_DIR, exist_ok=True)
    if not os.listdir(REAL_IMG_DIR):
        for i, (img, _) in enumerate(tqdm(test_dataset, desc="Lưu ảnh thật")):
            if i >= 10000: break
            save_image(img.repeat(3, 1, 1), os.path.join(REAL_IMG_DIR, f"real_{i}.png"))

    generated_count = 0
    while generated_count < 10000:
        if model_name in ['VAE', 'WAE-MMD']: z = torch.randn(config['batch_size'], config['latent_dim']).to(device)
        elif model_name == 'S-VAE': z = HypersphericalUniform(config['latent_dim'], device=device).sample((config['batch_size'],))
        elif model_name == 'VaDE':
            pi_dist = torch.distributions.Categorical(F.softmax(model.pi, dim=0))
            c = pi_dist.sample((config['batch_size'],))
            mu_c, log_var_c = model.mu_c[c], model.log_var_c[c]
            z = model.reparameterize(mu_c, log_var_c)

        gen_images = model.decoder(z)
        for i in range(gen_images.size(0)):
            if generated_count >= 10000: break
            save_image(gen_images[i].cpu().repeat(3, 1, 1), os.path.join(GEN_IMG_DIR, f"gen_{generated_count}.png"))
            generated_count += 1

    fid_value = fid_score.calculate_fid_given_paths([REAL_IMG_DIR, GEN_IMG_DIR], 50, device, 2048)
    return {"FID": fid_value, "LPIPS": final_lpips, "SSIM": final_ssim, "PSNR": final_psnr, "ACC": acc_score, "NMI": nmi_score, "ARI": ari_score}

# ==============================================================================
# --- BƯỚC 5: VÒNG LẶP HUẤN LUYỆN VÀ ĐÁNH GIÁ CHÍNH ---
# ==============================================================================
print("\n>>> Bước 5: Bắt đầu vòng lặp huấn luyện và đánh giá chính...")

models_to_run = {
    "VAE": VAE(CONFIG["latent_dim"]),
    "WAE-MMD": WAE_MMD(CONFIG["latent_dim"]),
    # "S-VAE": S_VAE(CONFIG["latent_dim"]),
    "VaDE": VaDE(CONFIG["latent_dim"], CONFIG["n_classes"])
}
all_results = {}

for model_name, model in models_to_run.items():
    print(f"\n{'='*30}\n Bắt đầu huấn luyện model: {model_name} \n{'='*30}")
    model.to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=CONFIG["lr"])

    for epoch in range(CONFIG["epochs"]):
        model.train()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['epochs']} [{model_name}]")
        for data, _ in pbar:
            data = data.to(DEVICE)
            optimizer.zero_grad()

            if model_name == 'VAE':
                recon_batch, mu, log_var = model(data)
                loss = model.loss_function(recon_batch, data, mu, log_var)
            elif model_name == 'WAE-MMD':
                recon_batch, z = model(data)
                loss = model.loss_function(recon_batch, data, z)
            elif model_name == 'S-VAE':
                recon_batch, q_z, p_z = model(data)
                loss = model.loss_function(recon_batch, data, q_z, p_z)
            elif model_name == 'VaDE':
                recon_batch, mu, log_var, z = model(data)
                loss = model.loss_function(recon_batch, data, mu, log_var, z)

            loss.backward()
            optimizer.step()
            pbar.set_postfix({"Loss": loss.item() / len(data)})

    torch.save(model.state_dict(), os.path.join(RESULTS_DIR, f"{model_name}.pth"))
    all_results[model_name] = evaluate_model(model, model_name, test_loader, DEVICE, CONFIG)

# ==============================================================================
# --- BƯỚC 6: IN BẢNG SO SÁNH CUỐI CÙNG ---
# ==============================================================================
print("\n>>> Bước 6: Hoàn tất! In bảng so sánh cuối cùng...")
your_model_results = {
    "FID": 15.5977, "LPIPS": 0.0454, "SSIM": 0.8717,
    "PSNR": 18.1812, "ACC": 0.9620, "NMI": 0.9026, "ARI": 0.9182
}
all_results["CS-WAE (Yours)"] = your_model_results
df = pd.DataFrame(all_results).T
df = df[["ACC", "NMI", "ARI", "FID", "LPIPS", "SSIM", "PSNR"]]

print("\n\n" + " BẢNG SO SÁNH HIỆU NĂNG TỔNG THỂ ".center(80, "="))
print(df.to_markdown(floatfmt=".4f"))
print("=" * 80)


>>> Bước 1: Cài đặt và import thư viện...

>>> Bước 2: Thiết lập cấu hình chung...

>>> Bước 3: Định nghĩa kiến trúc các model baseline...

>>> Bước 4: Xây dựng hàm đánh giá toàn diện...

>>> Bước 5: Bắt đầu vòng lặp huấn luyện và đánh giá chính...

 Bắt đầu huấn luyện model: VAE 


Epoch 50/50 [VAE]: 100%|██████████| 469/469 [00:10<00:00, 44.00it/s, Loss=96.4]



--- Bắt đầu đánh giá model: VAE ---
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.11/dist-packages/lpips/weights/v0.1/vgg.pth


Đánh giá VAE: 100%|██████████| 79/79 [00:03<00:00, 25.98it/s]


Bắt đầu tính FID cho VAE...


...


--- Bắt đầu đánh giá model: VaDE ---
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.11/dist-packages/lpips/weights/v0.1/vgg.pth


Đánh giá VaDE: 100%|██████████| 79/79 [00:03<00:00, 24.57it/s]


Bắt đầu tính FID cho VaDE...


100%|██████████| 200/200 [00:40<00:00,  4.96it/s]



>>> Bước 6: Hoàn tất! In bảng so sánh cuối cùng...


======================= BẢNG SO SÁNH HIỆU NĂNG TỔNG THỂ ========================
|                |    ACC |    NMI |    ARI |      FID |   LPIPS |   SSIM |    PSNR |
|:---------------|-------:|-------:|-------:|---------:|--------:|-------:|--------:|
| VAE            | 0.8119 | 0.6761 | 0.6478 |  19.1638 |  0.0564 | 0.9026 | 19.5457 |
| WAE-MMD        | 0.5039 | 0.4878 | 0.3589 | 123.9503 |  0.0229 | 0.9743 | 24.9512 |
| VaDE           | 0.6308 | 0.5744 | 0.4755 |  18.1685 |  0.0530 | 0.9090 | 19.8218 |
| CS-WAE (Yours) | 0.9620 | 0.9026 | 0.9182 |  15.5977 |  0.0454 | 0.8717 | 18.1812 |
